In [1]:
from streamlit import success

from useful.helpers import *
%matplotlib tk
seed = 42
jax.config.update("jax_enable_x64", True)
key = jax.random.PRNGKey(seed)

wigner_function_from_inference, t, f = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")

def tukey_window_this(mat, alpha=0.2):
    mat = np.asarray(mat)

    if mat.ndim == 1:
        # 1D case
        w = tukey(mat.size, alpha=alpha)
        return mat * w
    elif mat.ndim == 2:
        # 2D case
        ny, nx = mat.shape
        wx = tukey(nx, alpha=alpha)
        wy = tukey(ny, alpha=alpha)
        window2d = wy[:, None] * wx[None, :]
        return mat * window2d
    else:
        raise ValueError("Input must be 1D or 2D array")

wigner_function_from_inference = jnp.fft.fftshift(wigner_function_from_inference)
f = jnp.fft.fftshift(f)

wigner = tukey_window_this(smooth_matrix(wigner_function_from_inference, 5))

## Direct inversion

Let $n$ be the 2d windowed noise:

$$ n_{xy} = (F^{-1} \sqrt{p_n} \Xi)_{xy},$$

where $\Xi$ is 3d iid. Then,

$$ \sqrt{p_n}^{xy} = (\tilde{n} \Xi^{-1})^{xy},$$

since $\Xi$ is invertible almost surely.

In [84]:
def get_asd_with_force(image, itr=1):
    N = image.shape[0]
    image_ft = np.fft.fft2(image, norm="ortho")

    asd2d_samples = []
    big_xi_samples = []

    for i in range(itr):
        print(f"Drawing as2d {i} out of", itr)
        Xi = np.random.standard_normal(size=(N, N)) * 1e-3
        big_xi_samples.append(Xi)

        inversion_sucess = False
        while not inversion_sucess:
            try:
                Xi_inv = np.linalg.inv(Xi)
                inversion_sucess = True
            except np.linalg.LinAlgError:
                print("matrix was not invertible, drawing new xi")

        asd2d_sample = image_ft @ Xi_inv
        asd2d_samples.append(asd2d_sample)

    print("Calculating mean of the as2d...")
    asd2d = np.mean(np.array(asd2d_samples), axis=0)
    print("Done")
    return asd2d, asd2d_samples, big_xi_samples


def sample_from_amplitude_spectrum(asd, custom_big_xi=None):
    N = asd.shape[0]
    if custom_big_xi is None:
        print("Drawing new Xi")
        Xi = np.random.standard_normal(size=(N, N))
    else:
        print("Using given Xi")
        Xi = custom_big_xi

    to_ifft = asd @ Xi
    return np.fft.ifft2(to_ifft, norm="ortho")


In [3]:
white_noise_matrices = []
white_noise_matrices_smoothed = []
for i in range(1):
    white_noise = np.random.standard_normal(len(f))
    white_noise_stress, _, _ = Stress_re(white_noise, time=t, supress_print=True)
    white_noise_stress = white_noise_stress.real
    print(f"Calculated {i}th white noise stress")

    white_stress = tukey_window_this(white_noise_stress)
    white_noise_matrices.append(white_stress)

    smooth_white_stress = tukey_window_this(smooth_matrix(white_noise_stress, smoothing_lvl=5, mode="gaussian"))
    white_noise_matrices_smoothed.append(smooth_white_stress)

Calculated 0th white noise stress


In [85]:
# smooth_wnm = white_noise_matrices_smoothed[0]
normal_wnm = white_noise_matrices[0]
asd, asd_samples, Xi_samples = get_asd_with_force(normal_wnm, itr=1)

Drawing as2d 0 out of 1
Calculating mean of the as2d...
Done


In [90]:
N = asd.shape[0]
some_Xi = Xi_samples[0] + 1e-4 * np.random.standard_normal(size=(N, N))
some_asd = asd_samples[0]

forward_modeled_swnm = sample_from_amplitude_spectrum(asd=some_asd, custom_big_xi=some_Xi)

Using given Xi


In [91]:
# visualize_stress(smooth_wnm, rows=f, cols=t)
visualize_stress(forward_modeled_swnm, rows=f, cols=t, smooth=True)
# visualize_stress(np.log(asd_samples[5]), rows=f, cols=t)

In [94]:

visualize_stress(jnp.abs(wigner), rows=f, cols=t, smooth=False)

In [110]:
def Stress_re_TEST(xi, time, supress_print=False, downsample=False):
    """
    See also nifty8 `Stress` function.

    :param xi: jnp.array        A field to calculate the wigner function for. Either of complex or real data type.
                                If complex, assumed to be in DFT standard order (DC first, then positives then negatives).
    :param time: jnp.array      The real-space time array at which xi (or its iFFT if complex) was sampled at.
    :param supress_print: bool, Print imaginary part diagonstics (Wigner function should be real).
    :return:
    """

    FFT = lambda x, ax=-1: jnp.fft.fft(x, norm="ortho", axis=ax)
    iFFT = lambda x, ax=-1: jnp.fft.ifft(x, norm="ortho", axis=ax)

    if jnp.iscomplexobj(xi):
        xi = iFFT(xi)  # go to real space

    if downsample:
        step = 2
        xi = xi[::step]
        time = time[::step]

    t0 = time[0]
    dt = time[1]-time[0]
    N = len(xi)
    f = jnp.fft.fftfreq(N, d=dt)
    k = f.copy()
    df = f[1] - f[0]
    t = jnp.arange(N) / (N*df)  # dual time, equal to input time - time[0].


    if not supress_print:
        print("\nCalculating stress...")

    t_c = t[:, None]  # time cast
    k_c = k[None, :]  # shift frequencies cast
    xi_c = xi[:, None]  # xi values cast as rows

    if not supress_print:
        print("\t Calculating zeta plus")
    zeta_plus = jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta minus")
    zeta_minus = jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta plus in Fourier space")
    tilde_zeta_plus = FFT(zeta_plus, ax=0)

    if not supress_print:
        print("\t Calculating zeta minus in Fourier space")
    tilde_zeta_minus = FFT(zeta_minus, ax=0)

    if not supress_print:
        print("\t Calculating Phi matrix")
    Phi = jnp.abs(tilde_zeta_plus * tilde_zeta_minus.conj())  # domain = (h_space, h_space)

    if not supress_print:
        print("\t Inverse Fourier-Transforming columns of Phi matrix")
    S = iFFT(Phi, ax=1)
    S.block_until_ready()

    if not supress_print:
        print("\t ... Done")
    if not supress_print:
        diagnostic = jnp.abs(jnp.mean(S.imag))
        tmp = float(diagnostic)
        if diagnostic < 1e-10:
            print(f"\u2714 Mean imaginary part of stress field is smaller than 1e-10 threshold ({diagnostic}) ")
        else:
            raise_warning(
                f"Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 ({diagnostic}).")
    return S, t+t0, f

white_noise = np.random.standard_normal(len(f))
white_noise[2000:2500] = 100
print("times: ", t[2000:2500])
print("times: ", f[2000:2500])
white_noise_stress, _, _ = Stress_re_TEST(white_noise, time=t, supress_print=True)


times:  [0.48828125 0.48852539 0.48876953 0.48901367 0.48925781 0.48950195
 0.48974609 0.48999023 0.49023438 0.49047852 0.49072266 0.4909668
 0.49121094 0.49145508 0.49169922 0.49194336 0.4921875  0.49243164
 0.49267578 0.49291992 0.49316406 0.4934082  0.49365234 0.49389648
 0.49414062 0.49438477 0.49462891 0.49487305 0.49511719 0.49536133
 0.49560547 0.49584961 0.49609375 0.49633789 0.49658203 0.49682617
 0.49707031 0.49731445 0.49755859 0.49780273 0.49804688 0.49829102
 0.49853516 0.4987793  0.49902344 0.49926758 0.49951172 0.49975586
 0.5        0.50024414 0.50048828 0.50073242 0.50097656 0.5012207
 0.50146484 0.50170898 0.50195312 0.50219727 0.50244141 0.50268555
 0.50292969 0.50317383 0.50341797 0.50366211 0.50390625 0.50415039
 0.50439453 0.50463867 0.50488281 0.50512695 0.50537109 0.50561523
 0.50585938 0.50610352 0.50634766 0.5065918  0.50683594 0.50708008
 0.50732422 0.50756836 0.5078125  0.50805664 0.50830078 0.50854492
 0.50878906 0.5090332  0.50927734 0.50952148 0.50976562 

In [113]:
visualize_stress(wigner_function_from_inference, rows=f, cols=t, smooth=True)